# **1. Data cleaning and geographic enrichment**

This notebook keeps players born in Brazil, standardizes their birthplace fields, matches cities to Brazilian municipalities, and exports analysis-ready tables plus the JavaScript data files consumed by the HTML visualization. Foreign-born players are excluded from the treated dataset but counted in the audit.

In [1]:
from __future__ import annotations

import gzip
import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REFERENCE = RAW / "reference"
HTML_DATA = ROOT / "html" / "data"
for directory in (REFERENCE, PROCESSED, HTML_DATA):
    directory.mkdir(parents=True, exist_ok=True)

SEASON = 2026
PLAYERS_PATH = PROCESSED / f"brasileirao_{SEASON}_player_birthplaces.csv"
PROFILE_CACHE = RAW / "transfermarkt-api"

REFERENCE_URLS = {
    "brazilian_municipalities.csv": "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv",
    "brazilian_states.csv": "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/estados.csv",
    "brazil_states.geojson": "https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR?formato=application/vnd.geo+json&qualidade=minima&intrarregiao=UF",
}


## **2. Geographic reference data**

Municipality coordinates are cached from the `municipios-brasileiros` reference table. State boundaries come from the IBGE geographic-mesh API. Existing files are reused so reruns remain reproducible and do not require a network connection.

In [2]:
def download_if_missing(filename: str, url: str) -> Path:
    path = REFERENCE / filename
    if path.exists() and path.stat().st_size > 100:
        return path
    response = requests.get(
        url,
        headers={"User-Agent": "academic-research/brazilian-league-player-birthplaces"},
        timeout=90,
    )
    response.raise_for_status()
    path.write_bytes(response.content)
    return path


reference_paths = {
    name: download_if_missing(name, url)
    for name, url in REFERENCE_URLS.items()
}
print({name: path.stat().st_size for name, path in reference_paths.items()})


{'brazilian_municipalities.csv': 390201, 'brazilian_states.csv': 1077, 'brazil_states.geojson': 98502}


## **3. Brazilian-born player selection**

Birth country, rather than nationality, defines the study universe. Transfermarkt's additional birthplace field is recovered from the raw profile cache because it frequently identifies the state and improves municipality matching.

In [3]:
def clean_text(value: Any) -> str | None:
    if value is None or pd.isna(value):
        return None
    cleaned = " ".join(unicodedata.normalize("NFC", str(value)).strip().split())
    return cleaned or None


def normalize_key(value: Any) -> str | None:
    cleaned = clean_text(value)
    if cleaned is None:
        return None
    folded = "".join(
        character
        for character in unicodedata.normalize("NFKD", cleaned.casefold())
        if not unicodedata.combining(character)
    )
    return re.sub(r"[^a-z0-9]+", " ", folded).strip() or None


players = pd.read_csv(PLAYERS_PATH, dtype={"player_id": "string", "club_ids": "string"})
state_rows: list[dict[str, Any]] = []
for path in sorted(PROFILE_CACHE.glob(f"players_{SEASON}_batch_*.json.gz")):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        profiles = json.load(handle)["payload"]["data"]
    for profile in profiles:
        birth = profile.get("birthPlaceDetails") or {}
        state_rows.append({
            "player_id": str(profile["id"]),
            "birth_state_raw": clean_text(birth.get("placeOfBirthAdditionalInfo")),
        })

states_from_profiles = pd.DataFrame(state_rows).drop_duplicates("player_id")
players = players.merge(states_from_profiles, on="player_id", how="left", validate="one_to_one")
for column in ["player_name", "full_name", "birth_city", "birth_state_raw", "clubs", "position_group", "position"]:
    players[column] = players[column].map(clean_text)

foreigners_removed = int(players.birth_country_fifa_code.ne("BRA").sum())
brazilian = players.loc[players.birth_country_fifa_code.eq("BRA")].copy()
brazilian["birth_city_raw"] = brazilian["birth_city"]
brazilian["city_key"] = brazilian.birth_city.map(normalize_key)
brazilian["state_key"] = brazilian.birth_state_raw.map(normalize_key).replace({
    "distrito federal do brasil": "distrito federal",
})
print(f"{len(players):,} collected players | {len(brazilian):,} born in Brazil | {foreigners_removed:,} foreign-born removed")


668 collected players | 510 born in Brazil | 158 foreign-born removed


## **4. Municipality matching**

Matching prioritizes exact city-state pairs, then city names that are unique nationally. A small, explicit override table handles spelling errors, districts reported as cities, and two ambiguous municipality names. Every rule is retained in `geocoding_rule` and no coordinates are invented for missing birth cities.

In [4]:
municipalities = pd.read_csv(reference_paths["brazilian_municipalities.csv"])
state_reference = pd.read_csv(reference_paths["brazilian_states.csv"], encoding="utf-8-sig")
municipalities = municipalities.merge(
    state_reference[["codigo_uf", "uf", "nome", "regiao"]],
    on="codigo_uf",
    how="left",
    validate="many_to_one",
    suffixes=("", "_estado"),
)
municipalities["city_key"] = municipalities.nome.map(normalize_key)
municipalities["state_key"] = municipalities.nome_estado.map(normalize_key)
municipalities_by_pair = {
    (row.city_key, row.state_key): row
    for row in municipalities.itertuples(index=False)
}
city_name_counts = municipalities.groupby("city_key").size()
unique_municipalities = {
    row.city_key: row
    for row in municipalities.itertuples(index=False)
    if city_name_counts.loc[row.city_key] == 1
}

# Source correction -> standardized municipality. Rules remain visible in the output.
MANUAL_GEOCODING = {
    ("santa barbara", "goias"): ("santa barbara de goias", "goias", "municipality name completed"),
    ("santo antonio do salto da onca", "rio grande do norte"): ("santo antonio", "rio grande do norte", "historical locality name"),
    ("jotaesse", "parana"): ("tupassi", "parana", "district assigned to municipality"),
    ("campo dos goytacazes", "rio de janeiro"): ("campos dos goytacazes", "rio de janeiro", "spelling correction"),
    ("itaperica da serra", "sao paulo"): ("itapecerica da serra", "sao paulo", "spelling correction"),
    ("sao mateus", "sao paulo"): ("sao paulo", "sao paulo", "district assigned to municipality"),
    ("presidente dutra", None): ("presidente dutra", "maranhao", "ambiguous city disambiguated"),
    ("alvorada", None): ("alvorada", "rio grande do sul", "ambiguous city disambiguated"),
}

geocoded_rows: list[dict[str, Any]] = []
for player in brazilian.to_dict(orient="records"):
    city_key = player.get("city_key")
    state_key = player.get("state_key")
    city_key = None if pd.isna(city_key) else city_key
    state_key = None if pd.isna(state_key) else state_key
    municipality = None
    rule = "unmatched"
    note = None

    override = MANUAL_GEOCODING.get((city_key, state_key)) or MANUAL_GEOCODING.get((city_key, None))
    if city_key is None:
        rule = "missing_birth_city"
    elif override:
        target_city, target_state, note = override
        municipality = municipalities_by_pair[(target_city, target_state)]
        rule = "manual_override"
    elif state_key is not None and (city_key, state_key) in municipalities_by_pair:
        municipality = municipalities_by_pair[(city_key, state_key)]
        rule = "city_state_exact"
    elif city_key in unique_municipalities:
        municipality = unique_municipalities[city_key]
        rule = "unique_city_exact" if state_key is None else "unique_city_state_conflict"
        if state_key is not None:
            note = "unique municipality used; source state conflicts with reference"

    result = dict(player)
    result["geocoding_rule"] = rule
    result["geocoding_note"] = note
    result["map_status"] = "unmapped"
    result.update({
        "municipality_id": None,
        "birth_city": clean_text(player.get("birth_city")),
        "birth_state": clean_text(player.get("birth_state_raw")),
        "birth_state_code": None,
        "birth_region": None,
        "latitude": None,
        "longitude": None,
    })
    if municipality is not None:
        result.update({
            "municipality_id": int(municipality.codigo_ibge),
            "birth_city": municipality.nome,
            "birth_state": municipality.nome_estado,
            "birth_state_code": municipality.uf,
            "birth_region": municipality.regiao,
            "latitude": float(municipality.latitude),
            "longitude": float(municipality.longitude),
            "map_status": "mapped",
        })
    geocoded_rows.append(result)

brazilian_clean = pd.DataFrame(geocoded_rows)
brazilian_clean["municipality_id"] = brazilian_clean["municipality_id"].astype("Int64")
assert brazilian_clean.player_id.is_unique
assert len(brazilian_clean) == 510, f"Unexpected Brazilian-born player count: {len(brazilian_clean)}"
assert brazilian_clean.map_status.eq("mapped").sum() == 503, "Municipality coverage changed; review matching rules"
assert brazilian_clean.loc[brazilian_clean.map_status.ne("mapped"), "geocoding_rule"].eq("missing_birth_city").all()
display(brazilian_clean.geocoding_rule.value_counts().rename("players").to_frame())


,players
geocoding_rule,
city_state_exact,384
unique_city_exact,110
manual_override,8
missing_birth_city,7
unique_city_state_conflict,1


## **5. Visualization tables**

The main table keeps one row per Brazilian-born player. A second table aggregates mapped players by municipality for proportional-symbol sizing and ranking. Counts are based on players, not player-club rows.

In [5]:
clean_columns = [
    "season", "competition_id", "player_id", "player_name", "full_name",
    "date_of_birth", "birth_city_raw", "birth_state_raw", "birth_city",
    "birth_state", "birth_state_code", "birth_region", "municipality_id",
    "latitude", "longitude", "clubs", "club_ids", "position_group",
    "position", "profile_url", "map_status", "geocoding_rule", "geocoding_note",
]
brazilian_clean = (
    brazilian_clean[clean_columns]
    .sort_values(["map_status", "birth_state_code", "birth_city", "player_name"], na_position="last")
    .reset_index(drop=True)
)
mapped = brazilian_clean.loc[brazilian_clean.map_status.eq("mapped")].copy()
cities = (
    mapped.groupby(
        ["municipality_id", "birth_city", "birth_state", "birth_state_code", "birth_region", "latitude", "longitude"],
        as_index=False,
    )
    .agg(
        player_count=("player_id", "nunique"),
        players=("player_name", lambda values: " | ".join(sorted(values))),
    )
    .sort_values(["player_count", "birth_city"], ascending=[False, True])
    .reset_index(drop=True)
)
cities.insert(0, "city_rank", range(1, len(cities) + 1))
assert int(cities.player_count.sum()) == len(mapped)
display(cities.head(15))


,city_rank,municipality_id,birth_city,birth_state,birth_state_code,birth_region,latitude,longitude,player_count,players
0,1,3550308,São Paulo,São Paulo,SP,Sudeste,-23.53290,-46.6395,69,Ademir | Alan Patrick | Allex | André | Arthur...
1,2,3304557,Rio de Janeiro,Rio de Janeiro,RJ,Sudeste,-22.91290,-43.2003,40,Allan | Bruno Gomes | Bruno Tubarão | Carvalhe...
2,3,3106200,Belo Horizonte,Minas Gerais,MG,Sudeste,-19.91020,-43.9266,14,Bernard | Breno Lopes | Bruno Henrique | Edu R...
3,4,2927408,Salvador,Bahia,BA,Nordeste,-12.97180,-38.5011,14,Anderson Pato | Danilo | Eduardo Melo | Eric R...
4,5,2304400,Fortaleza,Ceará,CE,Nordeste,-3.71664,-38.5423,13,Artur | Da Silva | Dyogo Alves | Erick Pulga |...
5,6,4314902,Porto Alegre,Rio Grande do Sul,RS,Sul,-30.03180,-51.2065,8,Edenilson | Eduardo Sasha | Jáderson | Léo Ort...
6,7,4106902,Curitiba,Paraná,PR,Sul,-25.41950,-49.2646,7,Doma | Dudu | Gustavo Silva | Léo Andrade | Lé...
7,8,3543402,Ribeirão Preto,São Paulo,SP,Sudeste,-21.16990,-47.8099,6,Alex | Bruno Leonardo | Eduardo | Léo Jardim |...
8,9,3509502,Campinas,São Paulo,SP,Sudeste,-22.90530,-47.0659,5,Elias Manoel | Gabriel | Gustavo Scarpa | Rich...
9,10,5208707,Goiânia,Goiás,GO,Centro-Oeste,-16.68640,-49.2643,5,Dudu | Gustavo Neves | JP | Pedro Cobra | Zé V...


## **6. Export and audit**

CSV outputs support further analysis. JavaScript files expose the same validated records through global variables so the root `index.html` works from a simple static server and has no runtime dependency on the Python environment.

In [6]:
generated_at = datetime.now(timezone.utc).isoformat()
audit = pd.DataFrame([
    {"metric": "collected_players", "value": len(players)},
    {"metric": "foreign_born_removed", "value": foreigners_removed},
    {"metric": "brazilian_born_players", "value": len(brazilian_clean)},
    {"metric": "mapped_players", "value": int(brazilian_clean.map_status.eq("mapped").sum())},
    {"metric": "unmapped_missing_birth_city", "value": int(brazilian_clean.map_status.ne("mapped").sum())},
    {"metric": "mapped_municipalities", "value": int(len(cities))},
    {"metric": "municipality_coverage_pct", "value": round(brazilian_clean.map_status.eq("mapped").mean() * 100, 2)},
])

brazilian_clean.to_csv(PROCESSED / f"brasileirao_{SEASON}_brazilian_player_birthplaces.csv", index=False)
cities.to_csv(PROCESSED / f"brasileirao_{SEASON}_birthplace_cities.csv", index=False)
audit.to_csv(PROCESSED / f"brasileirao_{SEASON}_birthplace_treatment_audit.csv", index=False)

site_players = mapped.rename(columns={"clubs": "club"})[[
    "player_id", "player_name", "full_name", "date_of_birth",
    "birth_city", "birth_state", "birth_state_code", "birth_region",
    "municipality_id", "latitude", "longitude", "club",
    "position_group", "position", "profile_url",
]].to_dict(orient="records")
site_payload = {
    "metadata": {
        "season": SEASON,
        "competition": "Campeonato Brasileiro Serie A",
        "collected_players": int(len(players)),
        "brazilian_born_players": int(len(brazilian_clean)),
        "foreign_born_removed": foreigners_removed,
        "mapped_players": int(len(mapped)),
        "unmapped_players": int(len(brazilian_clean) - len(mapped)),
        "mapped_municipalities": int(len(cities)),
        "generated_at_utc": generated_at,
        "source": "Transfermarkt; municipality coordinates from municipios-brasileiros; state boundaries from IBGE",
    },
    "players": site_players,
}
(HTML_DATA / "birthplaces.js").write_text(
    "window.BIRTHPLACE_DATA = " + json.dumps(site_payload, ensure_ascii=False, separators=(",", ":")) + ";\n",
    encoding="utf-8",
)

with open(reference_paths["brazil_states.geojson"], encoding="utf-8") as handle:
    states_geojson = json.load(handle)
state_by_code = {str(row.codigo_uf): row for row in state_reference.itertuples(index=False)}
for feature in states_geojson["features"]:
    state = state_by_code[str(feature["properties"]["codarea"])]
    feature["properties"].update({"uf": state.uf, "name": state.nome, "region": state.regiao})
    # IBGE rings are counterclockwise (RFC 7946); D3's spherical polygons
    # expect the opposite winding for features smaller than a hemisphere.
    geometry = feature["geometry"]
    if geometry["type"] == "Polygon":
        geometry["coordinates"] = [list(reversed(ring)) for ring in geometry["coordinates"]]
    elif geometry["type"] == "MultiPolygon":
        geometry["coordinates"] = [
            [list(reversed(ring)) for ring in polygon]
            for polygon in geometry["coordinates"]
        ]
(HTML_DATA / "brazil-states.js").write_text(
    "window.BRAZIL_STATES = " + json.dumps(states_geojson, ensure_ascii=False, separators=(",", ":")) + ";\n",
    encoding="utf-8",
)

display(audit)
print(f"Processed CSVs: {PROCESSED}")
print(f"Visualization data: {HTML_DATA}")


,metric,value
0,collected_players,668.00
1,foreign_born_removed,158.00
2,brazilian_born_players,510.00
3,mapped_players,503.00
4,unmapped_missing_birth_city,7.00
5,mapped_municipalities,244.00
6,municipality_coverage_pct,98.63


Processed CSVs: /Users/gabrielcarbinatto/Documents/GitHub/brazilian-league-2026-player-birthplaces/data/processed
Visualization data: /Users/gabrielcarbinatto/Documents/GitHub/brazilian-league-2026-player-birthplaces/html/data
